# scHiCAR → MERFISH MOp mapping — process & data structures

This notebook walks through how single-cell **scHiCAR** data is mapped onto the
**BICCN MERFISH** mouse primary motor cortex (MOp) spatial atlas with **Tangram**,
and shows the **input** and **output** data structure at every step.

```
INPUT                                   MAPPING                    OUTPUT
scHiCAR RNA (cells×genes) ─┐
scHiCAR 3D features        │  shared genes → Tangram               ad_map (scHiCAR×MERFISH prob.)
MERFISH RNA (cells×genes) ─┘  map_cells_to_space(cells,uniform) →  projected celltype / 3D in space
```
Note: Tangram aligns by **RNA only**; 3D features ride along via the mapping.


## Data scope — what scHiCAR data is included

This notebook uses the local **GSE305439 / mouse_brain_scHiCAR_1** dataset, not
all public scHiCAR data. The original local scHiCAR RNA object has 5,313 cells;
this walkthrough uses the MOp-matched cortical subset
`data/schicar/schicar_rna_mop.h5ad` (5,163 cells, 19 cortical types), generated
by dropping non-MOp/non-cortical labels from the broader frontal-cortex sample.

The paired scHiCAR inputs used here are the GSE305439 RNA, DNA contact pairs,
and ATAC file plus derived per-cell feature matrices. **GSE267126 /
mouse_brain_scHiCAR_2 is not used** in this project because its DNA data are
multi-TB scale and were intentionally not downloaded. Therefore the outputs
below should be read as a projection of the GSE305439 scHiCAR_1 MOp-matched
subset, not as a comprehensive spatialization of every scHiCAR dataset.


In [1]:
import logging, numpy as np, pandas as pd, scanpy as sc, tangram as tg
logging.getLogger().setLevel(logging.WARNING); sc.settings.verbosity = 0
ROOT = "/Volumes/Extreme SSD/U-Chrom_explore_hicar"
print("scanpy", sc.__version__, "| tangram", tg.__version__)

scanpy 1.11.5 | tangram 1.0.4


## 1. INPUT A — scHiCAR RNA (single cell, no space)

In [2]:
adata_sch = sc.read_h5ad(f"{ROOT}/data/schicar/schicar_rna_mop.h5ad")
adata_sch

AnnData object with n_obs × n_vars = 5163 × 28783
    obs: 'celltype', 'DNAbarcode', 'nCount_RNA', 'nFeature_RNA', 'UMAP1', 'UMAP2'
    var: 'gene_id'
    obsm: 'X_umap'

In [3]:
print("shape (cells × genes):", adata_sch.shape)
print("obs columns:", list(adata_sch.obs.columns))
print("X:", type(adata_sch.X).__name__, adata_sch.X.dtype,
      "| raw counts? min/max =", float(adata_sch.X.min()), float(adata_sch.X.max()))
adata_sch.obs[["celltype", "DNAbarcode", "nCount_RNA", "nFeature_RNA"]].head()

shape (cells × genes): (5163, 28783)
obs columns: ['celltype', 'DNAbarcode', 'nCount_RNA', 'nFeature_RNA', 'UMAP1', 'UMAP2']
X: csr_matrix int64 | raw counts? min/max = 0.0 3492.0


,celltype,DNAbarcode,nCount_RNA,nFeature_RNA
ACGAACAAACTCAGACTT,Astro,ACGAACAAACTCAAGTCT,12792,3329
ACGAACAAACTCATTTGG,Sst,ACGAACAAACTCCCAAAT,9519,2705
ACGAACAAACTCCGTATT,L23IT.3,ACGAACAAACTCAATACG,14418,3285
ACGAACAAACTCGATAGG,Sst,ACGAACAAACTCCCTATC,28503,4522
ACGAACAAACTCTATCGC,Astro,ACGAACAAACTCGCGATA,8161,2413


In [4]:
# .var = genes (mouse symbols); .obs.celltype = the 19 cortical types
print("genes e.g.:", list(adata_sch.var_names[:8]))
adata_sch.obs["celltype"].value_counts().head(6)

genes e.g.: ['Gm26206', 'Xkr4', 'Gm18956', 'Gm1992', 'Gm7341', 'Gm10568', 'Gm37381', 'Rp1']


celltype
L5IT       642
L6CT       618
L23IT.1    575
Astro      407
L45IT      407
L6IT       373
Name: count, dtype: int64

## 2. INPUT B — MERFISH MOp (spatial, no 3D genome)

In [5]:
adata_mer = sc.read_h5ad(f"{ROOT}/data/merfish_mop/merfish_mouse2sample1.h5ad")
adata_mer

AnnData object with n_obs × n_vars = 13864 × 254
    obs: 'center_x', 'center_y', 'slice_id', 'sample_id', 'subclass', 'class_label'
    obsm: 'spatial'

In [6]:
print("shape (cells × genes):", adata_mer.shape, "  (254-gene panel)")
print("obs columns:", list(adata_mer.obs.columns))
print("obsm['spatial'] shape:", adata_mer.obsm["spatial"].shape, " = (x, y) centroids in µm")
adata_mer.obs[["subclass", "slice_id", "center_x", "center_y"]].head()

shape (cells × genes): (13864, 254)   (254-gene panel)
obs columns: ['center_x', 'center_y', 'slice_id', 'sample_id', 'subclass', 'class_label']
obsm['spatial'] shape: (13864, 2)  = (x, y) centroids in µm


,subclass,slice_id,center_x,center_y
index,,,,
100019614084014214413331344619872059390,L4/5 IT,mouse2_slice61,-4406.866048,-3784.263422
100024126192764500990464846159908804485,L4/5 IT,mouse2_slice1,-5938.214987,1334.835803
100036459431810739646600756615083690692,L4/5 IT,mouse2_slice300,1207.536506,2999.896805
100036464372885187596454120358701331853,L5 ET,mouse2_slice300,2121.430284,2490.716746
100042375868071099085129438220019302596,Oligo,mouse2_slice61,-4863.292523,-3079.352587


## 3. INPUT C — scHiCAR per-cell 3D-genome features (the payload to project)

In [7]:
hic = pd.read_csv(f"{ROOT}/data/schicar/hic_features.csv", index_col=0)
print("3D feature table shape (cells × features):", hic.shape)
print("index = scHiCAR RNA barcodes ; columns =", list(hic.columns))
hic.head()

3D feature table shape (cells × features): (5313, 8)
index = scHiCAR RNA barcodes ; columns = ['n_contacts', 'cis_frac', 'trans_frac', 'short_range_frac', 'long_range_frac', 'mean_log10_dist', 'pofs_slope', 'compartment_strength']


,n_contacts,cis_frac,trans_frac,short_range_frac,long_range_frac,mean_log10_dist,pofs_slope,compartment_strength
ACGAACAAACTCAGACTT,55403,0.747884,0.252116,0.710824,0.289176,5.899293,0.082503,0.903309
ACGAACAAACTCATTTGG,28122,0.606785,0.393215,0.804383,0.195617,5.669585,0.030094,0.925926
ACGAACAAACTCCGTATT,30234,0.744724,0.255276,0.820972,0.179028,5.540802,0.006347,0.932045
ACGAACAAACTCGATAGG,29638,0.703725,0.296275,0.874862,0.125138,5.370662,-0.038619,0.948406
ACGAACAAACTCTATCGC,25631,0.739651,0.260349,0.721120,0.278880,5.871076,0.075855,0.903576


## 4. Align inputs — shared genes
Tangram trains on genes present in **both** datasets (the bridge).

In [8]:
adata_sch.var_names = [g.lower() for g in adata_sch.var_names]; adata_sch.var_names_make_unique()
adata_mer.var_names = [g.lower() for g in adata_mer.var_names]; adata_mer.var_names_make_unique()
shared = sorted(set(adata_sch.var_names) & set(adata_mer.var_names))
print(f"shared genes: {len(shared)} / {adata_mer.n_vars} MERFISH panel")
print("examples:", shared[:12])

shared genes: 248 / 254 MERFISH panel
examples: ['1700022i11rik', '1810046k07rik', '5031425f14rik', '5730522e02rik', 'acta2', 'adam2', 'adamts2', 'adamts4', 'adra1b', 'alk', 'ankfn1', 'ano4']


## 5. Normalize + `tg.pp_adatas`

In [9]:
sc.pp.normalize_total(adata_sch, target_sum=1e4); sc.pp.log1p(adata_sch)
sc.pp.log1p(adata_mer)                      # MERFISH already volume-normalized
tg.pp_adatas(adata_sch, adata_mer, genes=shared)
print("training genes saved in .uns['training_genes']:", len(adata_sch.uns["training_genes"]))

training genes saved in .uns['training_genes']: 247


## 6. MAPPING — `tg.map_cells_to_space`
`mode='cells'`, `density_prior='uniform'` (single-cell MERFISH). `num_epochs=300`
here for a fast demo (production used 1000).

In [10]:
ad_map = tg.map_cells_to_space(
    adata_sch, adata_mer, mode="cells",
    density_prior="uniform", num_epochs=300, device="cpu")

Score: 0.443, KL reg: 0.000


Score: 0.879, KL reg: 0.005


Score: 0.889, KL reg: 0.004


## 7. OUTPUT ① — the mapping matrix `ad_map`
`ad_map` is an AnnData whose **rows = scHiCAR cells**, **columns = MERFISH cells**,
and `ad_map.X[i, j] = P(scHiCAR cell i is located at MERFISH cell j)`.

In [11]:
ad_map

AnnData object with n_obs × n_vars = 5163 × 13864
    obs: 'celltype', 'DNAbarcode', 'nCount_RNA', 'nFeature_RNA', 'UMAP1', 'UMAP2'
    var: 'center_x', 'center_y', 'slice_id', 'sample_id', 'subclass', 'class_label', 'uniform_density', 'rna_count_based_density'
    uns: 'train_genes_df', 'training_history'

In [12]:
M = ad_map.X
print("ad_map.shape (scHiCAR cells × MERFISH cells):", ad_map.shape)
print("M dtype:", M.dtype)
print("each scHiCAR cell's row sums to ~1 (a prob. distribution over space):",
      np.round(M.sum(1)[:5], 3))
print("a 3×5 corner of M:\n", np.round(M[:3, :5], 5))
print(f"max prob per scHiCAR cell: median {np.median(M.max(1)):.4f} "
      f"(uniform baseline = {1/ad_map.n_vars:.2e})")

ad_map.shape (scHiCAR cells × MERFISH cells): (5163, 13864)
M dtype: float32
each scHiCAR cell's row sums to ~1 (a prob. distribution over space): [1. 1. 1. 1. 1.]
a 3×5 corner of M:
 [[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
max prob per scHiCAR cell: median 0.0383 (uniform baseline = 7.21e-05)


## 8. OUTPUT ② — project cell-type labels into space
`project_cell_annotations` pushes any scHiCAR per-cell label through `M` to give,
for every MERFISH cell, a distribution over scHiCAR cell types.

In [13]:
tg.project_cell_annotations(ad_map, adata_mer, annotation="celltype")
pred = adata_mer.obsm["tangram_ct_pred"]
print("obsm['tangram_ct_pred'] shape (MERFISH cells × scHiCAR celltypes):", pred.shape)
adata_mer.obs["pred_celltype"] = pred.idxmax(axis=1).values
adata_mer.obs[["subclass", "pred_celltype"]].head()

obsm['tangram_ct_pred'] shape (MERFISH cells × scHiCAR celltypes): (13864, 19)


,subclass,pred_celltype
index,,
100019614084014214413331344619872059390,L4/5 IT,L45IT
100024126192764500990464846159908804485,L4/5 IT,L5IT
100036459431810739646600756615083690692,L4/5 IT,L5IT
100036464372885187596454120358701331853,L5 ET,PT
100042375868071099085129438220019302596,Oligo,Oligo


## 9. OUTPUT ③ — project the 3D-genome features into space
The same matrix carries the per-cell 3D features:
`spatial[j] = Σ_i M[i,j]·feature[i] / Σ_i M[i,j]` (weighted average per MERFISH cell).

In [14]:
F = hic.reindex(ad_map.obs_names)                 # align to scHiCAR-cell axis of M
nan = F.isna().to_numpy()
num = M.T @ np.where(nan, 0.0, F.to_numpy())
den = M.T @ (~nan).astype(float)
proj = pd.DataFrame(num / den, index=adata_mer.obs_names,
                    columns=["hic_" + c for c in F.columns])
print("projected 3D features shape (MERFISH cells × features):", proj.shape)
proj.head()

projected 3D features shape (MERFISH cells × features): (13864, 8)


,hic_n_contacts,hic_cis_frac,hic_trans_frac,hic_short_range_frac,hic_long_range_frac,hic_mean_log10_dist,hic_pofs_slope,hic_compartment_strength
index,,,,,,,,
100019614084014214413331344619872059390,17015.860647,0.724784,0.275216,0.771454,0.228546,5.774847,0.060368,0.917371
100024126192764500990464846159908804485,24965.568340,0.767418,0.232582,0.792619,0.207381,5.706300,0.046350,0.923641
100036459431810739646600756615083690692,19449.549433,0.739545,0.260455,0.773589,0.226411,5.759509,0.057027,0.915279
100036464372885187596454120358701331853,25823.683195,0.740928,0.259072,0.778659,0.221341,5.727122,0.049933,0.918449
100042375868071099085129438220019302596,19200.942537,0.750332,0.249668,0.700946,0.299054,5.852146,0.072860,0.906568


## 10. INPUT → OUTPUT data-structure summary

| stage | object | shape | meaning |
|---|---|---|---|
| input A | `adata_sch` (AnnData) | cells × genes | scHiCAR RNA + `obs.celltype/DNAbarcode` |
| input B | `adata_mer` (AnnData) | cells × 254 | MERFISH RNA + `obsm['spatial']`, `obs.subclass` |
| input C | `hic` (DataFrame) | cells × 8 | scHiCAR per-cell 3D features |
| mapping | `ad_map` (AnnData) | **scHiCAR × MERFISH** | `X[i,j]=P(cell i → space j)` |
| output ② | `obsm['tangram_ct_pred']` | MERFISH × celltypes | projected cell-type probs |
| output ③ | `proj` (DataFrame) | **MERFISH × 8** | 3D features placed in space |

**Key point:** `ad_map` is learned from **RNA only**; cell types and 3D features
are projected *afterwards* through it. So anything placed in space is determined
by the RNA bridge (see project README / validation for what this does and does
not license scientifically).

---
## 11. Other omic in space ① — imputed RNA transcriptome
`tg.project_genes` pushes the **full** scHiCAR transcriptome through the same map
to predict spatial expression of genes (here top 2000 HVGs) — including genes not
on the MERFISH panel.

In [15]:
sch_full = sc.read_h5ad(f"{ROOT}/data/schicar/schicar_rna_mop.h5ad")
sc.pp.normalize_total(sch_full, target_sum=1e4); sc.pp.log1p(sch_full)
sch_full = sch_full[ad_map.obs_names].copy()                 # align cells to the map
sc.pp.highly_variable_genes(sch_full, n_top_genes=2000)
hvg = list(sch_full.var_names[sch_full.var.highly_variable])
ad_ge = tg.project_genes(ad_map, sch_full[:, hvg].copy())    # MERFISH cells × HVG (imputed)
rna_imputed = np.asarray(ad_ge.X)
print("imputed RNA in space (MERFISH cells × HVG):", rna_imputed.shape)
pd.DataFrame(rna_imputed[:3, :5], columns=hvg[:5], index=adata_mer.obs_names[:3])

imputed RNA in space (MERFISH cells × HVG): (13864, 2000)


,Sox17,Gm37323,Gm37988,Rgs20,St18
index,,,,,
100019614084014214413331344619872059390,0.000049,0.000002,0.000655,0.193884,0.038147
100024126192764500990464846159908804485,0.000041,0.009404,0.003637,0.183037,0.023393
100036459431810739646600756615083690692,0.000019,0.000002,0.000650,0.183272,0.031160


### Note: RNA stored in the `.h5cd` is Tangram-projected RNA

The RNA matrix written to `cellm/rna_imputed_schicar_hvg` is **not** the raw
scHiCAR RNA count matrix. It is normalized/log1p scHiCAR MOp-subset RNA,
restricted to the top 2,000 HVGs, then projected through the Tangram `ad_map`
onto MERFISH cells. Rows are MERFISH cells; columns are scHiCAR HVG genes.


## 12. Other omic in space ② — scATAC accessibility
Per-cell scATAC features were aggregated from the fragments file by the **same
18bp barcode** that keys the 3D genome (DNAbarcode), then carried into space via
the **same RNA-derived `ad_map`**. So ATAC is handled exactly like the 3D genome:
the spatial position is set by RNA; ATAC just rides along by cell id.

In [16]:
z = np.load(f"{ROOT}/data/schicar/atac_features.npz", allow_pickle=True)
atac = pd.DataFrame(z["binned_cpm"], index=z["rna_barcode"]).reindex(ad_map.obs_names)
A = atac.to_numpy(float); mA = np.isnan(A)
atac_spatial = (M.T @ np.where(mA, 0.0, A)) / (M.T @ (~mA).astype(float))   # MERFISH × 1Mb bins
nf = pd.Series(z["n_fragments"], index=z["rna_barcode"]).reindex(ad_map.obs_names).to_numpy()
mN = np.isnan(nf)
nf_spatial = (M.T @ np.where(mN, 0.0, nf)) / (M.T @ (~mN).astype(float))
print("ATAC 1Mb-binned accessibility in space (MERFISH cells × bins):", atac_spatial.shape)
print("ATAC depth n_fragments projected:", nf_spatial.shape)
print(">> projected through ad_map (the RNA-derived map): SPATIAL POSITION COMES FROM RNA.")

ATAC 1Mb-binned accessibility in space (MERFISH cells × bins): (13864, 2645)
ATAC depth n_fragments projected: (13864,)
>> projected through ad_map (the RNA-derived map): SPATIAL POSITION COMES FROM RNA.


## 13. Three scHiCAR modalities now in MERFISH space
All three are positioned by the single RNA mapping; 3D & ATAC are attached by cell id.

In [17]:
print("RNA imputed     :", rna_imputed.shape)
print("ATAC binned     :", atac_spatial.shape, " (+ atac_n_fragments", nf_spatial.shape, ")")
print("(paper 5kb loops added in the next section, §12b)")

RNA imputed     : (13864, 2000)
ATAC binned     : (13864, 2645)  (+ atac_n_fragments (13864,) )
(paper 5kb loops added in the next section, §12b)


## 12b. 3D-genome features in space — paper 5kb chromatin loops
The paper provides two 5kb loop-call resources. We keep them separate:

- **Peakachu 5kb loops**: original loop pairs are stored as
  `results/loops_5kb_peakachu`; a 1Mb loop-anchor-density summary is stored as
  `tracks/paper_loop_density_5kb_peakachu`.
- **scDeepLUCIA 5kb loops**: original loop pairs are stored as
  `results/loops_5kb_scdeeplucia`; a 1Mb loop-anchor-density summary is stored
  as `tracks/paper_loop_density_5kb_scdeeplucia`.

The 1Mb summaries are **anchor densities**, not original 5kb loop-pair lists.
They exist only so loop-derived signals can align with the 1Mb ATAC bin axis.
The full `.h5cd` export is generated by scripts 24, 26, and 27.

In [18]:
zL = np.load(f"{ROOT}/data/schicar/paper_loop_density_5kb.npz", allow_pickle=True)
loop_ct = list(zL["cell_types"]); loop_dens = zL["density"]          # (22, bins)
paper2sch = {"Pvalb":"Pvalb","Sst":"Sst","Vip":"Vip","PT":"PT","L45IT":"L45IT",
             "L5IT":"L5IT","L6IT":"L6IT","L6CT":"L6CT","L56NP":"L56NP","OPC":"OPC",
             "Oligo":"Oligo","MGL":"MGL","L23IT-1":"L23IT.1","L23IT-2":"L23IT.2",
             "L23IT-3":"L23IT.3","Astrocyte":"Astro","Endothelial":"Endo"}
sch_to_row = {paper2sch[p]: i for i, p in enumerate(loop_ct) if p in paper2sch}
cell_ct = sc.read_h5ad(f"{ROOT}/data/schicar/schicar_rna_mop.h5ad").obs["celltype"].astype(str)
cell_ct = cell_ct.reindex(ad_map.obs_names)
F_loop = np.full((len(cell_ct), loop_dens.shape[1]), np.nan, dtype=float)
for j, ct in enumerate(cell_ct):
    if ct in sch_to_row:
        F_loop[j] = loop_dens[sch_to_row[ct]]
mL = np.isnan(F_loop)
loop_spatial = (M.T @ np.where(mL, 0.0, F_loop)) / (M.T @ (~mL).astype(float))   # MERFISH × bins
print("Peakachu 5kb loop-anchor density in space (MERFISH cells × 1Mb bins):", loop_spatial.shape)
print("scHiCAR types matched to Peakachu loops:", sorted(sch_to_row))
print(">> 1Mb track = 5kb loop anchor density summary, carried to space via RNA ad_map.")

Peakachu 5kb loop-anchor density in space (MERFISH cells × 1Mb bins): (13864, 2645)
scHiCAR types matched to Peakachu loops: ['Astro', 'Endo', 'L23IT.1', 'L23IT.2', 'L23IT.3', 'L45IT', 'L56NP', 'L5IT', 'L6CT', 'L6IT', 'MGL', 'OPC', 'Oligo', 'PT', 'Pvalb', 'Sst', 'Vip']
>> 1Mb track = 5kb loop anchor density summary, carried to space via RNA ad_map.


## 14. U-Chrom `ChromData` export with missing spot coordinates
Do **not** treat a MERFISH cell centroid as a U-Chrom `spot`.

In U-Chrom, a `spot` is a chromatin locus/bin with its own per-spot 3D coordinate.
A MERFISH centroid is cell-level spatial metadata: it belongs in `cells`, not in
`coords`/`spots`. Conceptually, one MERFISH cell would contain many chromatin
spots/bins.

Correct layout used by the generated `.h5cd`:
- `cells`: MERFISH cells, with `x_centroid`, `y_centroid`, `subclass`, etc.
- `spots`: 1Mb chromatin bins/loci inside cells, with `chrom`, `start`, `end`,
  `trace_id`, `cell_id`.
- `coords`: all `NaN`, explicitly marking missing per-bin 3D coordinates.
- `tracks`: projected per-spot 1Mb annotations: ATAC, Peakachu anchor density,
  and scDeepLUCIA anchor density.
- `cellm`: cell-level projected matrices, including `rna_imputed_schicar_hvg`
  (MERFISH cells × 2,000 scHiCAR HVGs) and projected ATAC depth.
- `results/loops_5kb_peakachu` and `results/loops_5kb_scdeeplucia`: original
  paper 5kb loop calls.


In [19]:
H5CD_NAN = f"{ROOT}/data/GSE305439_scHiCAR1_MOp_to_MERFISH_mouse2sample1_cellx1MbBin_ATAC_loop5kb_nanCoords.h5cd"
print("generated file:", H5CD_NAN)
print("writer/update scripts:")
print("  scripts/24_write_schicar_merfish_cellxbin_h5cd.py")
print("  scripts/26_parse_scdeeplucia_loops.py")
print("  scripts/27_update_h5cd_with_scdeeplucia_loops.py")
print("  scripts/28_append_schicar_rna_to_h5cd.py")
print("  scripts/29_write_spatialized_schicar_rna_h5ad.py")

generated file: /Volumes/Extreme SSD/U-Chrom_explore_hicar/data/GSE305439_scHiCAR1_MOp_to_MERFISH_mouse2sample1_cellx1MbBin_ATAC_loop5kb_nanCoords.h5cd
writer/update scripts:
  scripts/24_write_schicar_merfish_cellxbin_h5cd.py
  scripts/26_parse_scdeeplucia_loops.py
  scripts/27_update_h5cd_with_scdeeplucia_loops.py
  scripts/28_append_schicar_rna_to_h5cd.py
  scripts/29_write_spatialized_schicar_rna_h5ad.py


## 15. Read the written `.h5cd` structure

The file is large, so this cell inspects HDF5 metadata and small slices rather
than loading the full `ChromData` object into memory.


In [20]:
import h5py, os, numpy as np

H5CD = f"{ROOT}/data/GSE305439_scHiCAR1_MOp_to_MERFISH_mouse2sample1_cellx1MbBin_ATAC_loop5kb_nanCoords.h5cd"
with h5py.File(H5CD, "r") as h5:
    print("file:", H5CD)
    print("size MB:", round(os.path.getsize(H5CD) / 1e6, 1))
    print("top-level groups:", list(h5.keys()))
    print("coords:", h5["coords"].shape, h5["coords"].dtype,
          "sample all NaN:", bool(np.isnan(h5["coords"][:5]).all()))
    print("spots rows:", int(h5["spots"].attrs["_n_rows"]),
          "columns:", list(h5["spots"].attrs["_columns"]))
    print("cells rows:", int(h5["cells"].attrs["_n_rows"]),
          "columns:", list(h5["cells"].attrs["_columns"]))
    print("tracks:", {k: h5["tracks"][k].shape for k in h5["tracks"].keys()})
    print("cellm:", {k: h5["cellm"][k].shape for k in h5["cellm"].keys()})
    print("results:", {k: int(h5["results"][k].attrs["_n_rows"]) for k in h5["results"].keys()})
    print("RNA genes:", len(h5["uns"]["rna_imputed_schicar_hvg_genes"]),
          [g.decode() for g in h5["uns"]["rna_imputed_schicar_hvg_genes"][:5]])
    print("coordinate_status:", h5["uns"]["coordinate_status"][()].decode())
    print("rna_imputed_source:", h5["uns"]["rna_imputed_source"][()].decode())
    print("spatialized scHiCAR RNA companion:",
          h5["uns"]["source_schicar_rna_spatialized_h5ad"][()].decode())


file: /Volumes/Extreme SSD/U-Chrom_explore_hicar/data/GSE305439_scHiCAR1_MOp_to_MERFISH_mouse2sample1_cellx1MbBin_ATAC_loop5kb_nanCoords.h5cd
size MB: 1456.6
top-level groups: ['cellm', 'cells', 'coords', 'layers', 'results', 'spots', 'traces', 'tracks', 'uns']
coords: (36670280, 3) float64 sample all NaN: True
spots rows: 36670280 columns: ['chrom', 'start', 'end', 'trace_id', 'cell_id']
cells rows: 13864 columns: ['merfish_cell_id', 'x_centroid', 'y_centroid', 'subclass', 'class_label', 'slice_id', 'sample_id']
tracks: {'atac_binned_cpm': (36670280,), 'paper_loop_density_5kb_peakachu': (36670280,), 'paper_loop_density_5kb_scdeeplucia': (36670280,)}
cellm: {'atac_n_fragments_projected': (13864, 1), 'rna_imputed_schicar_hvg': (13864, 2000)}
results: {'loops_5kb_peakachu': 447625, 'loops_5kb_scdeeplucia': 1054101}
RNA genes: 2000 ['Sox17', 'Gm37323', 'Gm37988', 'Rgs20', 'St18']
coordinate_status: missing_per_bin_3d_coordinates_all_coords_nan
rna_imputed_source: scHiCAR GSE305439 MOp-sub

## 16. Read the companion raw scHiCAR RNA `.h5ad`

The main `.h5cd` uses MERFISH cells as its cell axis, so raw scHiCAR RNA cannot
be stored there as a normal cell matrix without mixing two different cell axes.

To keep the original scHiCAR RNA expression matrix, the companion `.h5ad` stores:
- `X`: original scHiCAR MOp-subset RNA matrix (`scHiCAR cells × genes`).
- `obsm['spatial']`: expected MERFISH centroid inferred from Tangram mapping
  probabilities.
- `obsm['spatial_argmax']`: centroid of the single most likely MERFISH cell.
- `obs`: original scHiCAR cell metadata plus Tangram match diagnostics.

This is **not** a directly measured spatial RNA experiment. The spatial fields
are inferred from the same Tangram map used elsewhere in this notebook.


In [21]:
import scanpy as sc

RNA_SPATIAL_H5AD = f"{ROOT}/data/GSE305439_scHiCAR1_MOp_RNA_with_Tangram_MERFISH_mouse2sample1_spatial.h5ad"
ad_rna_spatial = sc.read_h5ad(RNA_SPATIAL_H5AD)

print("file:", RNA_SPATIAL_H5AD)
print("shape (scHiCAR cells × genes):", ad_rna_spatial.shape)
print("obs columns added:", [
    c for c in ad_rna_spatial.obs.columns
    if c.startswith("tangram_") or c == "spatial_source"
])
print("obsm:", {k: ad_rna_spatial.obsm[k].shape for k in ad_rna_spatial.obsm.keys()})
print("spatialization note:", ad_rna_spatial.uns["spatialization"]["method"])
ad_rna_spatial.obs[[
    "celltype",
    "tangram_argmax_merfish_subclass",
    "tangram_max_prob",
    "tangram_effective_n_merfish_cells",
]].head()


file: /Volumes/Extreme SSD/U-Chrom_explore_hicar/data/GSE305439_scHiCAR1_MOp_RNA_with_Tangram_MERFISH_mouse2sample1_spatial.h5ad
shape (scHiCAR cells × genes): (5163, 28783)
obs columns added: ['tangram_argmax_merfish_cell_id', 'tangram_argmax_merfish_subclass', 'tangram_max_prob', 'tangram_effective_n_merfish_cells', 'tangram_entropy', 'spatial_source']
obsm: {'X_umap': (5163, 2), 'spatial': (5163, 2), 'spatial_argmax': (5163, 2)}
spatialization note: Tangram map_cells_to_space


,celltype,tangram_argmax_merfish_subclass,tangram_max_prob,tangram_effective_n_merfish_cells
ACGAACAAACTCAGACTT,Astro,Astro,0.028264,87.998413
ACGAACAAACTCATTTGG,Sst,Pvalb,0.059945,51.931881
ACGAACAAACTCCGTATT,L23IT.3,L2/3 IT,0.038929,62.534641
ACGAACAAACTCGATAGG,Sst,Sst,0.053447,54.145035
ACGAACAAACTCTATCGC,Astro,Astro,0.023173,84.285912


## 17. Honest scope notes
- The `.h5cd` keeps **two loop callers separately**: Peakachu and scDeepLUCIA.
- Original 5kb loop pairs are stored in `results/loops_5kb_peakachu` and
  `results/loops_5kb_scdeeplucia`.
- 1Mb tracks are only anchor-density summaries aligned to ATAC:
  `paper_loop_density_5kb_peakachu` and `paper_loop_density_5kb_scdeeplucia`.
- The 1Mb loop tracks are not 1Mb loop contact matrices and not substitutes for
  the original 5kb loop lists.
- The loops are cell-type/pseudobulk features carried to MERFISH space by the same
  RNA-derived Tangram map; they add no independent positional information.
- scHiCAR RNA is stored as `cellm/rna_imputed_schicar_hvg`: normalized/log1p
  MOp-subset scHiCAR RNA, top 2,000 HVGs, projected to MERFISH cells with the
  same `ad_map`.
- Raw scHiCAR RNA for the MOp-matched subset is preserved separately in
  `GSE305439_scHiCAR1_MOp_RNA_with_Tangram_MERFISH_mouse2sample1_spatial.h5ad`,
  with inferred MERFISH spatial coordinates and Tangram diagnostics.
- U-Chrom `coords` are all `NaN`: each spot is `MERFISH cell × 1Mb genome bin`,
  but the per-bin 3D coordinate is missing. MERFISH centroids remain cell-level
  metadata, not chromatin spot coordinates.